# Temporal spectral embedding of HAR volatility features

**Research question.** Do realized-volatility dynamics fall into a small number
of recurring *regimes*, and can an unsupervised manifold method recover them
and forecast next-period volatility?

**Approach.** Each trading day is represented by its 6-dim **HAR feature
vector** — rolling means of diurnal-adjusted RV at geometric lags
`[1, 5, 25, 125, 625, 3125]`. These already summarize multi-scale history, so
there is **no windowing and no residualization** — we embed the HAR vectors
directly. We then:

1. build a **spectral embedding** (graph-Laplacian eigenmaps) of the HAR
   vectors, and inspect what structure it captures;
2. wire up the full prediction path — embed a test day via Nyström, run a
   **kNN regression in embedding space** to forecast **next-period `adj_RV`**, and
   benchmark it against a direct HAR-space kNN and a persistence baseline.

This notebook is the **source of truth** for the embedding code: the `# export`
cells are auto-exported to `src/features/extractors/spectral_embedding.py`.

In [ ]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

REPO = Path.cwd()
while REPO.parent != REPO and not (REPO / "src").is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn.neighbors import KNeighborsRegressor

from src.backtest.executor import _build_har_and_calendar, load_and_transform
from src.features.transforms.scaling import rolling_robust_scale
from src.features.transforms.target import PERIODS_PER_DAY
from src.models.knn import gaussian_weights

## 1. Load data, HAR features, and the forecast target

`load_and_transform` builds the diurnal-adjusted target `adj_RV` (intraday
seasonality removed, 240-period winsorized); `_build_har_and_calendar` adds the
HAR rolling-mean lags. We keep only the six `har_ma_*` columns (calendar dummies
are not HAR features). The **forecast target** is `adj_RV` one period (the next 30-min bar)
ahead.

In [ ]:
df, _ = load_and_transform(
    "data", exog_cols=[],
    target_use_diurnal=True, target_winsor_window=240, dropna_with_exog=True,
)
df, feature_names = _build_har_and_calendar(df, exog_cols=[], add_calendar=True)
df["t"] = pd.to_datetime(df["t"])

har_cols = [c for c in feature_names if c.startswith("har_ma_")]
df["next_period_adj_RV"] = df["adj_RV"].shift(-1)   # forecast target
df = df.dropna(subset=har_cols + ["next_period_adj_RV"]).reset_index(drop=True)

print(f"rows:        {len(df):,}   date range: {df['t'].min()}  ..  {df['t'].max()}")
print(f"HAR features: {har_cols}")
print(f"target:       next-period adj_RV (+1 bar)")

## 2. The series we're modelling

`har_ma_1` (the shortest HAR lag ≈ current vol level) over time, with major
risk-off episodes shaded — the regime structure we hope the embedding will
organize.

In [ ]:
level_series = df["har_ma_1"].to_numpy()
fig, ax = plt.subplots(figsize=(13, 3.0))
ax.plot(df["t"], level_series, lw=0.3, c="k", alpha=0.6)
for label, lo, hi, color in [
    ("GFC", "2008-09-01", "2009-04-01", "tab:red"),
    ("EU debt", "2011-07-01", "2011-10-30", "tab:orange"),
    ("China devalue", "2015-08-15", "2015-10-01", "tab:purple"),
    ("Volmageddon", "2018-02-01", "2018-02-20", "tab:brown"),
    ("COVID", "2020-02-20", "2020-04-30", "tab:blue"),
    ("SVB / Mar23", "2023-03-08", "2023-03-22", "tab:green"),
]:
    ax.axvspan(pd.Timestamp(lo), pd.Timestamp(hi), alpha=0.18, color=color, label=label)
ax.legend(fontsize=7, ncol=6, loc="upper right")
ax.set_xlabel("date"); ax.set_ylabel("har_ma_1  (adj_RV units)")
ax.set_ylim(np.quantile(level_series, [0.001, 0.999]))
fig.suptitle(
    "Volatility level (har_ma_1) over time, 2005–2024\n"
    "Shaded = known risk-off episodes — one of the six HAR features fed to the embedding.",
    fontsize=10,
)
plt.tight_layout(); plt.show()

## 3. The spectral-embedding machinery  *(source of truth)*

The `# export` cells below define the module shipped to
`src/features/extractors/spectral_embedding.py`:

- `build_embedding(views, d, k_graph)` — k-NN affinity graph + normalized
  Laplacian + bottom-`d` eigenvectors, via `sklearn.manifold.SpectralEmbedding`.
- `SpectralBasis` — holds `phi_train` and a `NearestNeighbors` index; `.embed(v)`
  extends a *new* point via weighted-kNN Nyström (sklearn ships no `.transform()`).

Edit the embedding here, not in `src/`.

In [ ]:
# export
"""Temporal spectral embedding (sklearn-backed).

A thin wrapper over :class:`sklearn.manifold.SpectralEmbedding` that gives
:class:`~src.backtest.multi_stage.MultiStageBacktest` the
``.phi_train + .embed(v_test)`` interface it expects. Out-of-sample
extension is a weighted-kNN Nyström over the training views.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from sklearn.manifold import SpectralEmbedding
from sklearn.neighbors import NearestNeighbors

In [ ]:
# export
@dataclass
class SpectralBasis:
    """Frozen training-side spectral embedding state.

    Holds the training-side embedding ``phi_train`` plus a fitted
    ``NearestNeighbors`` index over the training views (the NN index
    already retains the views internally in ``_fit_X``).
    ``sklearn.manifold.SpectralEmbedding`` doesn't implement
    ``transform()``, so the NN index drives weighted-kNN Nyström
    extension for new test points.
    """

    phi_train: np.ndarray
    k_graph: int
    nn_index: NearestNeighbors

    def embed(self, v_test: np.ndarray) -> np.ndarray:
        """Embed a single test view via weighted-kNN Nyström. Returns shape (d,)."""
        dists, idx = self.nn_index.kneighbors(v_test[None, :], n_neighbors=self.k_graph)
        dists = dists.ravel()
        idx = idx.ravel()
        sigma = float(np.median(dists)) + 1e-12
        w = np.exp(-(dists**2) / (2 * sigma**2))
        s = w.sum()
        if s <= 0:
            return self.phi_train[idx].mean(axis=0)
        w = w / s
        return (w[:, None] * self.phi_train[idx]).sum(axis=0)

    def embed_batch(self, V_test: np.ndarray) -> np.ndarray:
        """Embed a batch of test views. Returns shape (M, d). Vectorized."""
        dists, idx = self.nn_index.kneighbors(V_test, n_neighbors=self.k_graph)
        sigma = np.median(dists, axis=1, keepdims=True) + 1e-12
        w = np.exp(-(dists**2) / (2 * sigma**2))
        w = w / np.clip(w.sum(axis=1, keepdims=True), 1e-12, None)
        return np.einsum("mk,mkd->md", w, self.phi_train[idx])

In [ ]:
# export
def build_embedding(views: np.ndarray, d: int, k_graph: int, seed: int = 42) -> SpectralBasis:
    """Spectral embedding of temporal views.

    Delegates to :class:`sklearn.manifold.SpectralEmbedding` with binary
    ``affinity='nearest_neighbors'`` (k = ``k_graph``) and ARPACK
    eigensolver. The fitted ``NearestNeighbors`` index over ``views`` is
    cached on the returned :class:`SpectralBasis` so out-of-sample test
    points can be embedded via Nyström without re-fitting anything.
    """
    spectral = SpectralEmbedding(
        n_components=d,
        affinity="nearest_neighbors",
        n_neighbors=k_graph,
        random_state=seed,
    )
    phi_train = spectral.fit_transform(views)
    nn_index = NearestNeighbors(n_neighbors=k_graph).fit(views)
    return SpectralBasis(
        phi_train=phi_train,
        k_graph=k_graph,
        nn_index=nn_index,
    )

## 4. Build the embedding

Each point is one day's 6-dim HAR vector. We per-column rolling-robust-scale the
HAR lags (so the noisy `har_ma_1` doesn't dominate the k-NN distance), subsample
to one point per trading day, and embed. No windowing — the HAR lags already
encode history.

In [ ]:
WARMUP_DAYS = 500
train_win = WARMUP_DAYS * PERIODS_PER_DAY
SUBSAMPLE_STEP = PERIODS_PER_DAY   # one point per trading day
EMBEDDING_DIM = 8
GRAPH_K = 10
NEIGHBOR_K = 25                    # for the kNN regression in §7

X_har = df[har_cols].to_numpy(dtype=np.float64)
X_har_scaled = rolling_robust_scale(X_har, train_win)

# Subsample from the warmup boundary onward so the scaler is initialized.
idx = np.arange(train_win, len(df), SUBSAMPLE_STEP)
views       = X_har_scaled[idx]                              # (N, 6) — embedding input
target      = df["next_period_adj_RV"].to_numpy()[idx]         # (N,)   — forecast target
current_rv  = df["adj_RV"].to_numpy()[idx]                  # (N,)   — for persistence baseline
view_dates  = df["t"].iloc[idx].reset_index(drop=True)
view_rms    = df["har_ma_1"].to_numpy()[idx]               # vol-level proxy for coloring
ma125       = df["har_ma_125"].to_numpy()[idx]             # MA baseline (paper's naive)

print(f"views shape: {views.shape}   (N days x {views.shape[1]} HAR features)")
print(f"date span:   {view_dates.iloc[0]}  ..  {view_dates.iloc[-1]}")

In [ ]:
%time basis = build_embedding(views, d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)
phi = basis.phi_train
print(f"phi: {phi.shape}")

## 5. What does the embedding capture?

Color the 2-D embedding by **calendar year** and by **vol level** (`har_ma_1`).
A regime embedding would organize by market state; an amplitude-driven one
shows a smooth RMS gradient.

*(This embedding is fit on **all** days, in-sample, for visualization. The out-of-sample forecast in §7 uses a strict walk-forward.)*

In [ ]:
view_years = view_dates.dt.year.to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
sc0 = axes[0].scatter(phi[:, 0], phi[:, 1], c=view_years, s=8, alpha=0.7, cmap="viridis")
axes[0].set_xlabel("φ₁  (spectral coordinate 1)"); axes[0].set_ylabel("φ₂  (spectral coordinate 2)")
axes[0].set_title("colored by calendar year")
plt.colorbar(sc0, ax=axes[0], label="year")

sc1 = axes[1].scatter(phi[:, 0], phi[:, 1], c=view_rms, s=8, alpha=0.7, cmap="plasma", norm=LogNorm())
axes[1].set_xlabel("φ₁  (spectral coordinate 1)"); axes[1].set_ylabel("φ₂  (spectral coordinate 2)")
axes[1].set_title("colored by vol level (har_ma_1, log)")
plt.colorbar(sc1, ax=axes[1], label="har_ma_1")

fig.suptitle(
    "Spectral embedding of per-day HAR feature vectors  (each point = one period (the next 30-min bar))\n"
    "Finding so far: φ₁ tracks volatility level, years intermingle — a 1-D vol continuum, not discrete regimes.",
    fontsize=10,
)
plt.tight_layout(); plt.show()

In [ ]:
WINDOWS = [
    ("GFC",          "2008-09-01", "2009-04-01", "tab:red"),
    ("EU debt",      "2011-07-01", "2011-10-30", "tab:orange"),
    ("China devalue","2015-08-15", "2015-10-01", "tab:purple"),
    ("Volmageddon",  "2018-02-01", "2018-02-20", "tab:brown"),
    ("COVID",        "2020-02-20", "2020-04-30", "tab:blue"),
    ("SVB / Mar23",  "2023-03-08", "2023-03-22", "tab:green"),
]
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(phi[:, 0], phi[:, 1], c="lightgray", s=6, alpha=0.5, label="all days")
for label, lo, hi, color in WINDOWS:
    mask = (view_dates >= pd.Timestamp(lo)) & (view_dates < pd.Timestamp(hi))
    if mask.sum() == 0:
        continue
    ax.scatter(phi[mask, 0], phi[mask, 1], c=color, s=24, alpha=0.95,
               edgecolors="k", linewidths=0.4, label=f"{label}  (n={mask.sum()})")
ax.set_xlabel("φ₁  (spectral coordinate 1)"); ax.set_ylabel("φ₂  (spectral coordinate 2)")
fig.suptitle(
    "Where do known crises land?\n"
    "Distinct episodes smear along the high-amplitude tail rather than forming separated blobs.",
    fontsize=10,
)
ax.legend(fontsize=8, loc="best")
plt.tight_layout(); plt.show()

## 6. Higher embedding dimensions

Do `φ₃`/`φ₄` add structure beyond `φ₁`/`φ₂`, or just noise?

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for i in range(3):
    for j in range(3):
        ax = axes[i, j]
        if i <= j:
            ax.set_visible(False)
            continue
        ax.scatter(phi[:, j], phi[:, i + 1], c=view_years, cmap="viridis", s=4, alpha=0.6)
        ax.set_xlabel(f"φ_{j + 1}"); ax.set_ylabel(f"φ_{i + 2}")
fig.suptitle("First four embedding dimensions (lower-triangle pairs), colored by year", y=1.01, fontsize=10)
plt.tight_layout(); plt.show()

## 7. Walk-forward forecast of next-bar `adj_RV`

Matching the paper's walk-forward protocol. At each test day *t* the embedding
is **rebuilt on the trailing `TRAIN_DAYS` window** of HAR vectors; the test day
is Nyström-extended into that embedding; a Gaussian-weighted kNN in φ-space
regresses the target (next-bar `adj_RV`). The embedding refits every
`REFIT_EVERY` days — rebuild cost makes per-step refit impractical, so Nyström
extends test days between refits (the same amortization the production
`spectral_knn` backtest uses via `refit_frequency`).

Four benchmarks, all on the same walk-forward schedule:

| Model | Predicts next-bar adj_RV from |
|---|---|
| **spectral kNN** | kNN in φ-space (the embedding) |
| **direct HAR kNN** | kNN in raw 6-dim HAR space (no embedding — the ablation) |
| **persistence** | current `adj_RV` |
| **MA baseline** | `har_ma_125` (the paper's naive benchmark) |

OOS R² is reported **relative to the MA baseline** (`1 − MSE/MSE_MA`) — the same
definition as the paper's tables, so it's directly comparable to the Ridge-HAR
72%. (The embedding forces one-view-per-day subsampling, so this is a daily
proxy of the paper's all-bars eval.)

In [ ]:
TRAIN_DAYS  = WARMUP_DAYS   # 500 trading days of context (= paper's 24000-bar window, subsampled)
REFIT_EVERY = 5            # rebuild the embedding every 5 test days (~weekly; matches production cadence)

n = len(views)
n_test = n - TRAIN_DAYS
pred_phi  = np.empty(n_test)
pred_view = np.empty(n_test)

basis_wf = knn_phi = knn_view = None
t0 = time.perf_counter()
for i in range(n_test):
    t = TRAIN_DAYS + i
    tr = slice(t - TRAIN_DAYS, t)
    if i % REFIT_EVERY == 0:                                   # refit embedding + regressors
        basis_wf = build_embedding(views[tr], d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)
        knn_phi  = KNeighborsRegressor(n_neighbors=NEIGHBOR_K, weights=gaussian_weights).fit(basis_wf.phi_train, target[tr])
        knn_view = KNeighborsRegressor(n_neighbors=NEIGHBOR_K, weights=gaussian_weights).fit(views[tr], target[tr])
    pred_phi[i]  = knn_phi.predict(basis_wf.embed(views[t])[None, :])[0]   # Nystrom-extend, then kNN in phi
    pred_view[i] = knn_view.predict(views[t][None, :])[0]                  # direct HAR-space kNN
print(f"walk-forward: {n_test} test days, refit every {REFIT_EVERY} days, {time.perf_counter() - t0:.1f}s")

y_te         = target[TRAIN_DAYS:]
pred_persist = current_rv[TRAIN_DAYS:]
pred_ma      = ma125[TRAIN_DAYS:]
dates_te     = view_dates.iloc[TRAIN_DAYS:].to_numpy()

mse_ma = float(np.mean((pred_ma - y_te) ** 2))
def _scores(pred):
    mse = float(np.mean((pred - y_te) ** 2))
    return mse, 1.0 - mse / float(np.var(y_te)), 1.0 - mse / mse_ma   # MSE, R2(var), R2(vs MA)

print()
print(f"{'model':18s}{'MSE':>11s}{'R2(var)':>10s}{'R2(vs MA)':>11s}")
for name, pred in [("spectral kNN (phi)", pred_phi), ("direct HAR kNN", pred_view),
                   ("persistence", pred_persist), ("MA baseline", pred_ma)]:
    mse, r2v, r2m = _scores(pred)
    print(f"  {name:16s}{mse:11.6f}{r2v:+10.4f}{r2m:+11.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(y_te, pred_phi, s=6, alpha=0.4, c="tab:blue")
lim = [0, float(np.quantile(y_te, 0.999))]
axes[0].plot(lim, lim, "k--", lw=0.8, label="perfect")
axes[0].set_xlim(lim); axes[0].set_ylim(lim)
axes[0].set_xlabel("actual next-bar adj_RV"); axes[0].set_ylabel("predicted (spectral kNN)")
axes[0].set_title("walk-forward spectral kNN: predicted vs actual")
axes[0].legend(fontsize=8)

axes[1].plot(dates_te, y_te, lw=0.5, c="k", alpha=0.5, label="actual")
axes[1].plot(dates_te, pred_phi, lw=0.5, c="tab:blue", alpha=0.7, label="spectral kNN")
axes[1].plot(dates_te, pred_ma, lw=0.5, c="tab:orange", alpha=0.6, label="MA baseline")
axes[1].set_xlabel("date"); axes[1].set_ylabel("next-bar adj_RV")
axes[1].set_title("walk-forward forecast (test period)")
axes[1].legend(fontsize=8)
fig.suptitle(
    'Walk-forward next-bar adj_RV forecast via kNN in the HAR embedding\nR² is vs the MA baseline (paper convention). If spectral ≈ direct HAR kNN, the embedding adds no value.',
    fontsize=10,
)
plt.tight_layout(); plt.show()

## 8. Nyström out-of-sample extension — how accurate is `embed`?

The forecast above relies on `basis_tr.embed_batch` to place test days into the
train embedding. Sanity-check that extension: hold out 10% of all days, fit the
embedding on the rest, Nyström-extend the held-out days, and compare to where
they land in a full-set embedding (Procrustes-aligned via the shared training
points, since spectral coordinates are defined only up to rotation/sign).

In [ ]:
from scipy.linalg import orthogonal_procrustes

rng = np.random.default_rng(0)
n = len(views)
test_i = rng.choice(n, size=int(0.10 * n), replace=False)
train_i = np.setdiff1d(np.arange(n), test_i)

b_train = build_embedding(views[train_i], d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)
b_full = build_embedding(views, d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)

A = b_full.phi_train[train_i]
B = b_train.phi_train
R, _ = orthogonal_procrustes(A, B)               # align train-only frame -> full frame

phi_nystrom = b_train.embed_batch(views[test_i]) @ R
phi_truth = b_full.phi_train[test_i]
err = np.linalg.norm(phi_nystrom - phi_truth, axis=1)
scale = b_full.phi_train.std(axis=0).mean()

print(f"held-out days: {len(test_i)}")
print(f"Nyström vs full-embedding per-point distance (Procrustes-aligned):")
print(f"  median = {np.median(err):.4f}   90th = {np.quantile(err, 0.9):.4f}   max = {err.max():.4f}")
print(f"  embedding scale = {scale:.4f}   ->  relative median = {np.median(err)/scale*100:.1f}% of scale")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.4))
axes[0].scatter(A[:, 0], A[:, 1], c="lightgray", s=4, alpha=0.4, label="train")
axes[0].scatter(phi_truth[:, 0], phi_truth[:, 1], c="tab:blue", s=12, alpha=0.7, label="held-out (truth)")
axes[0].set_title("full-set embedding"); axes[0].legend(fontsize=8)
axes[1].scatter(A[:, 0], A[:, 1], c="lightgray", s=4, alpha=0.4, label="train")
axes[1].scatter(phi_nystrom[:, 0], phi_nystrom[:, 1], c="tab:red", s=12, alpha=0.7, label="held-out (Nyström)")
axes[1].set_title("train-only + Nyström extension"); axes[1].legend(fontsize=8)
for ax in axes:
    ax.set_xlabel("φ₁"); ax.set_ylabel("φ₂")
fig.suptitle("Nyström out-of-sample extension — truth vs extended (held-out days)", y=1.01, fontsize=10)
plt.tight_layout(); plt.show()

## Findings

1. **The HAR-feature embedding encodes volatility level, not discrete regimes.**
   φ₁ tracks `har_ma_1`; calendar years intermingle and named crises smear
   along the amplitude tail (§5). The structure is a 1-D vol continuum.

2. **The embedding adds nothing — and at this horizon kNN is the wrong model.**
   Walk-forward (paper protocol: rebuild the embedding on the trailing 500-day
   window, refit every 5 days; OOS R² reported vs the MA baseline). Forecasting
   the next 30-min bar: spectral kNN R² ≈ +0.17 and direct HAR-space kNN
   R² ≈ +0.18 — indistinguishable, embedding a hair *worse*. Both are beaten by
   **persistence** (R² ≈ +0.29): at the 30-min horizon vol is so autocorrelated
   that “next ≈ current” dominates. And both sit far below the paper's
   **Ridge–HAR (72%)** — kNN's neighbour-averaging smooths away the persistence
   signal Ridge captures linearly. The spectral step is a lossy
   re-parameterization (φ₁ ≈ vol level ⇒ φ-neighbours = HAR-neighbours), so it
   re-discovers the neighbourhood the raw kNN already uses.

3. **Nyström extension is usable but imprecise** (§8): held-out per-point error
   is a sizeable fraction of the embedding scale, consistent with a noisy
   manifold rather than crisp clusters.

### Implication

Regime identity is plausibly a **cross-channel** property (how RV co-moves with
returns, sentiment, VIX), not a single-series HAR shape — motivating the
correlation-matrix fingerprint of Papenbrock & Schwendner (2015) as the next
variant. The kNN-in-embedding forecast also wants a scale-aware target if the
embedding ever becomes shape- rather than level-driven.